### Importaciones

In [ ]:
import os
import pytz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from datetime import datetime
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

## Input ruta de archivo

In [ ]:
# Input ruta de archivo y eliminar las comillas
ruta_data = input("Introduce la ruta del archivo de datos (data.csv): ").strip().strip('"')

# Deducción automática del archivo de metadatos y aviso si no existe
ruta_metadata = ruta_data.replace("data.csv", "metadata.csv")
if not os.path.exists(ruta_metadata):
    raise FileNotFoundError(f"No se encontró metadata.csv en la ruta:\n{ruta_metadata}")

# Carga de datos
data = pd.read_csv(ruta_data)
metadata = pd.read_csv(ruta_metadata)

## Validación y conversión de fechas

In [ ]:
# 1. Verificar que la columna 'Date Time' existe
if 'Date Time' not in data.columns:
    raise KeyError("La columna 'Date Time' no está presente en el archivo de datos.")

# 2. Pasar todo a texto para unificar formato
dt_str = data['Date Time'].astype(str)

# 3. Eliminar la parte de zona horaria al final (+02:00, +0200, -01:00, Z, etc.)
#    Nos quedamos solo con la fecha y la hora "local" tal cual aparecen
dt_str = dt_str.str.replace(r'(\+|-)\d{2}:?\d{2}$', '', regex=True)
dt_str = dt_str.str.replace(r'Z$', '', regex=True)

# 4. Convertir a datetime SIN zona horaria
data['Date Time'] = pd.to_datetime(dt_str, errors='coerce')

# 5. Comprobar si hay fechas no convertibles y decidir qué hacer
n_invalidas = data['Date Time'].isna().sum()
if n_invalidas > 0:
    print(f"Advertencia: {n_invalidas} filas con 'Date Time' no válida. Se eliminarán.")
    data = data.dropna(subset=['Date Time']).copy()

# 6. Selección de columnas relevantes
columnas_necesarias = ['Speed(km/hour)', 'Segment ID', 'Corridor/Region Name', 'Date Time']
data = data[columnas_necesarias].copy()

# 7. Metadata
metadata = metadata[['Segment ID', 'Segment Length(Kilometers)']].copy()

print("Conversión de fechas completada. Formato sin zona horaria, horas locales conservadas.")
print(f"Registros válidos: {len(data)}")


## Input periodos de tiempo

In [ ]:
# Input periodos de tiempo
def solicitar_periodos_por_rango():
    print("Introduce periodos en formato MM/DD/YYYY - MM/DD/YYYY")
    print("Presiona Enter sin escribir nada para finalizar.")

    periodos = {} # Diccionario para almacenar los periodos válidos
    while True:
        entrada = input("Periodo: ").strip()
        if not entrada:
            break # Si está vacío, salir del bucle

        try: # Separar la cadena por el guion
            partes = entrada.split('-')
            if len(partes) != 2:
                print("Formato incorrecto. Usa: MM/DD/YYYY - MM/DD/YYYY")
                continue

            # Limpiar y convertir las fechas
            inicio_str, fin_str = partes[0].strip(), partes[1].strip()
            inicio = datetime.strptime(inicio_str, "%m/%d/%Y")
            fin = datetime.strptime(fin_str, "%m/%d/%Y")

            # Validar orden cronológico
            if inicio >= fin:
                print("La fecha de inicio debe ser anterior a la de fin.")
                continue

            # Formatear el nombre del periodo con salto de línea
            nombre_periodo = f"{inicio.strftime('%d %b %Y')}\n{fin.strftime('%d %b %Y')}"
            periodos[nombre_periodo] = (inicio, fin)

        # Capturar errores de formato de fecha
        except ValueError as e:
            print(f"Error en las fechas: {e}")
            continue

    return periodos # Devolver todos los periodos introducidos

# Entrada de periodos
periodos_personalizados = solicitar_periodos_por_rango()

# Generar periodo completo si no se introdujo ninguno
if not periodos_personalizados:
    print("No se introdujeron periodos. Se usará el rango completo del dataset.")

    if data.empty:
        print("El dataset está vacío. No se puede generar un periodo automático.")
    else:
        inicio_auto = data['Date Time'].min()
        fin_auto = data['Date Time'].max()

        nombre_periodo_auto = f"{inicio_auto.strftime('%d %b %Y')}\n{fin_auto.strftime('%d %b %Y')}"
        periodos_personalizados[nombre_periodo_auto] = (inicio_auto, fin_auto)

## Calcular V₈₅

In [ ]:
# Calcular V₈₅

def calcular_v85_ponderada(df_segmento, metadata):
    # Calcular V₈₅ individual por segmento
    v85_segmentos = df_segmento.groupby('Segment ID')['Speed(km/hour)'] \
        .apply(lambda x: np.percentile(x, 85)).reset_index(name='V85_segmento')
    
    # Unir longitudes desde metadata (por 'Segment ID') y calcular Longitud/V85_segmento
    v85_segmentos = v85_segmentos.merge(metadata, on='Segment ID', how='left')
    v85_segmentos['Longitud/V85'] = v85_segmentos['Segment Length(Kilometers)'] / v85_segmentos['V85_segmento']
    
    # Añadir nombre del tramo ('Corridor/Region Name') y renombrar columnas
    v85_segmentos = v85_segmentos.merge(
        df_segmento[['Segment ID', 'Corridor/Region Name']],
        on='Segment ID', how='left'
    ).drop_duplicates(subset='Segment ID')

    v85_segmentos = v85_segmentos.rename(columns={
        'Segment Length(Kilometers)': 'Longitud',
        'Corridor/Region Name': 'Tramo'
    })

    # Agrupar por tramo, calcular longitud total por tramo y V85 ponderada y longitud total por tramo
    longitud_total = v85_segmentos.groupby('Tramo')['Longitud'].sum()
    suma_l_div_v85 = v85_segmentos.groupby('Tramo')['Longitud/V85'].sum()
    v85_ponderada = longitud_total / suma_l_div_v85

    # Preparar DataFrame de salida
    resultado = pd.DataFrame({
        'Tramo': longitud_total.index,
        'Longitud': longitud_total.values,
        'V85': v85_ponderada.values
    })

    return resultado

## Generar tabla

In [ ]:
## Generar tabla con V85 ponderada por tramo y por periodo

tabla_final = None       # Aquí se irán acumulando las columnas de V85 por periodo
longitudes = None        # Aquí se guarda la longitud total por tramo (solo se calcula una vez)

# Ordenar los periodos por fecha de inicio
periodos_ordenados = sorted(periodos_personalizados.items(), key=lambda x: x[1][0])

# Asegurar que la columna 'Date Time' no tenga zona horaria (offset-aware)
if pd.api.types.is_datetime64tz_dtype(data['Date Time']):
    data['Date Time'] = data['Date Time'].dt.tz_localize(None)

# Recorrer todos los periodos definidos por el usuario
for nombre_columna, (inicio, fin) in periodos_ordenados:
    # Eliminar zona horaria también en los límites del periodo, si la tuvieran
    if getattr(inicio, 'tzinfo', None) is not None:
        inicio = inicio.tz_localize(None)
    if getattr(fin, 'tzinfo', None) is not None:
        fin = fin.tz_localize(None)

    # Filtrar las filas del periodo actual
    mask = (data['Date Time'] >= inicio) & (data['Date Time'] <= fin)
    df_periodo = data.loc[mask].copy()

    if df_periodo.empty:
        print(f"Aviso: El periodo {nombre_columna} no tiene datos. Se omite.")
        continue

    # Calcular V85 ponderada para este periodo
    resultado = calcular_v85_ponderada(df_periodo, metadata)

    # Guardar la longitud total por tramo (solo la primera vez)
    if longitudes is None:
        longitudes = resultado[['Tramo', 'Longitud']].set_index('Tramo')

    # Preparar columna V85 para este periodo (etiquetada con las fechas)
    v85 = resultado[['Tramo', 'V85']].set_index('Tramo').rename(columns={'V85': nombre_columna})

    # Unir esta columna a la tabla final (por 'Tramo')
    if tabla_final is None:
        tabla_final = v85
    else:
        tabla_final = tabla_final.join(v85, how='outer')

# Añadir columna de longitud (desde 'longitudes') y convertir 'Tramo' en columna visible
tabla_final = longitudes.join(tabla_final)
tabla_final = tabla_final.reset_index()  # Mueve 'Tramo' al cuerpo de la tabla

# Redondear Longitud a 3 decimales
tabla_final['Longitud'] = tabla_final['Longitud'].round(3)

# Redondear columnas de V85 a 2 decimales (todas menos 'Tramo' y 'Longitud')
v85_columnas = tabla_final.columns.difference(['Tramo', 'Longitud'])
tabla_final[v85_columnas] = tabla_final[v85_columnas].round(2)


## Explicacion

In [ ]:
explicacion = """

**V₈₅ ponderada por tramo y periodo**

La **V₈₅** mostrada a continuación no es el percentil 85 directo de todas las velocidades del tramo (lo que se conoce como **V₈₅ global**),  
sino una **V₈₅ ponderada por la longitud de los segmentos** que lo componen.

Esta medida tiene en cuenta tanto la velocidad como el peso de cada segmento en función de su longitud, y es más representativa  
cuando los segmentos tienen tamaños muy distintos. Su valor se aproxima al tiempo que tomaría recorrer un tramo si se circulara  
a la V85 de cada segmento.

En resumen:

- **V₈₅ global**: percentil 85 de todas las velocidades del tramo (no usada aquí).
- **V₈₅ (mostrada abajo)**: combina V85 de cada segmento según su longitud.

Cada celda indica la V₈₅ ponderada del tramo en un periodo específico.

También se incluye la **longitud total del tramo** en kilómetros.

La V₈₅ ponderada se calcula del siguiente modo:

$V_{85} = \\frac{\\sum L}{\\sum \\left(\\frac{L}{V_{85,\\text{segmento}}}\\right)}$

Donde:
- $L$ es la longitud del segmento
- $V_{85,\\text{segmento}}$ es el percentil 85 de velocidad para cada segmento
- El resultado representa una velocidad tipo "media armónica inversa", más representativa de la movilidad en tramos largos o irregulares.
"""
display(Markdown(explicacion))

## Resultados

In [ ]:
# Resultados 
# Formatear visualmente las columnas con decimales
formato_columnas = {col: "{:.2f}" for col in v85_columnas}
formato_columnas["Longitud"] = "{:.3f}"

tabla_formateada = (
    tabla_final
    .style
    .set_table_styles([
        {'selector': 'th', 'props': [('white-space', 'pre-line')]}
    ])
    .format(formato_columnas)
)

display(tabla_formateada)

### Graficos diarios agrupados

In [ ]:
# 1) SOLICITAR GRANULARIDAD

def solicitar_granularidad():
    print("Selecciona la granularidad para el análisis de velocidad:")
    print("Ejemplos: 15min, 30min, 60min, 5min...")
    print("Enter = 15min por defecto")
    
    entrada = input("Granularidad: ").strip().lower()

    # Valor por defecto
    if entrada == "":
        print("Usando granularidad por defecto: 15min")
        return "15min"

    # Caso: introduce solo número (ej: "30")
    if entrada.isdigit():
        minutos = int(entrada)
        return f"{minutos}min"

    # Validación con pandas
    try:
        pd.date_range("2025-01-01", periods=2, freq=entrada)
        return entrada
    except ValueError:
        print("Granularidad no válida → usando 15min")
        return "15min"


granularidad_usuario = solicitar_granularidad()

# 2) PREPARAR DATOS PARA GRÁFICOS

colores_periodo = [
    {'relleno': 'skyblue',   'linea': 'blue'},
    {'relleno': '#FFB347',   'linea': '#FF8C00'}, #naranja
    {'relleno': 'lightgreen','linea': 'green'},
    {'relleno': 'plum',      'linea': 'purple'}
]

data_grafico = data.copy()

if pd.api.types.is_datetime64tz_dtype(data_grafico['Date Time']):
    data_grafico['Date Time'] = data_grafico['Date Time'].dt.tz_localize(None)

data_grafico['HoraFranja'] = data_grafico['Date Time'].dt.floor(granularidad_usuario)
data_grafico['HoraDecimal'] = (
    data_grafico['HoraFranja'].dt.hour +
    data_grafico['HoraFranja'].dt.minute/60
)

# 3) CALCULAR ESTADÍSTICAS HORARIAS

def calcular_estadisticas_horarias_precisas(data, periodos):
    resultados = []

    for nombre_periodo, (inicio, fin) in sorted(periodos.items(), key=lambda x: x[1][0]):

        if getattr(inicio, 'tzinfo', None):
            inicio = inicio.tz_localize(None)
        if getattr(fin, 'tzinfo', None):
            fin = fin.tz_localize(None)

        df_periodo = data[(data['Date Time'] >= inicio) & (data['Date Time'] <= fin)]
        if df_periodo.empty:
            continue

        for tramo in sorted(df_periodo['Corridor/Region Name'].unique()):
            df_tramo = df_periodo[df_periodo['Corridor/Region Name'] == tramo]

            resumen = df_tramo.groupby('HoraDecimal')['Speed(km/hour)'].agg(
                p5=lambda x: np.percentile(x, 5),
                v85=lambda x: np.percentile(x, 85),
                p95=lambda x: np.percentile(x, 95)
            ).reset_index()

            resumen["Periodo"] = nombre_periodo
            resumen["Tramo"] = tramo
            resultados.append(resumen)

    return pd.concat(resultados, ignore_index=True)


estadisticas_horarias = calcular_estadisticas_horarias_precisas(data_grafico, periodos_personalizados)
estadisticas_horarias.sort_values(["Tramo", "Periodo"], inplace=True)

# 4) PREGUNTAR SI SE QUIERE VELOCIDAD LÍMITE


resp_vlim = input("\n¿Incluir la velocidad límite en los gráficos? (s/n): ").strip().lower()

usar_vlim = (resp_vlim == "s")
vlimites = {}

if usar_vlim:
    tramos = sorted(estadisticas_horarias["Tramo"].unique())
    print("\n¿Todos los tramos tienen la misma velocidad límite? (s/n)")
    comun = input("> ").strip().lower()

    if comun == "s":
        v = float(input("\nVelocidad límite común (km/h): "))
        for t in tramos:
            vlimites[t] = v
    else:
        print("\nIntroduce velocidad límite por tramo:")
        for t in tramos:
            while True:
                try:
                    v = float(input(f"V. límite para '{t}': "))
                    vlimites[t] = v
                    break
                except ValueError:
                    print("Introduce un número válido.")



# 5) GRAFICAR TODOS LOS TRAMOS

for tramo, grupo_tramo in estadisticas_horarias.groupby("Tramo"):
    plt.figure(figsize=(10, 4))

    periodos_ordenados = list(grupo_tramo["Periodo"].unique())
    legend_elements = []

    print(f"\n=== Tramo: {tramo} ===")

    for i, periodo in enumerate(periodos_ordenados):
        df = grupo_tramo[grupo_tramo["Periodo"] == periodo]
        color = colores_periodo[i % len(colores_periodo)]
        periodo_plano = periodo.replace("\n", " - ")

        # --- Sombreado P5-P95 ---
        plt.fill_between(df["HoraDecimal"], df["p5"], df["p95"],
                         color=color['relleno'], alpha=0.3)

        # --- Línea V85 ---
        plt.plot(df["HoraDecimal"], df["v85"],
                 color=color['linea'], linewidth=2)

        legend_elements.extend([
            Line2D([], [], linestyle='None', label=periodo_plano),
            Line2D([], [], color=color['relleno'], lw=8, alpha=0.3, label='Percentiles 5%-95%'),
            Line2D([], [], color=color['linea'], lw=2, label='V₈₅')
        ])


    #  LÍNEA DE VELOCIDAD LÍMITE (solo si usar_vlim = True)

    if usar_vlim:
        vlim = vlimites[tramo]

        # Línea roja horizontal
        plt.axhline(y=vlim, color="#CC0000", linewidth=1.0)

        # Texto a la derecha y ligeramente por encima
        x_text = grupo_tramo["HoraDecimal"].max() - 0.1
        y_text = vlim - 1.2

        plt.text(
            x_text, y_text,
            f"V lim = {vlim:.1f} km/h",
            color="#CC0000",
            fontsize=7,
            ha="right",
            va="top"
        )

    #  FORMATo DEL GRÁFICO

    plt.suptitle(tramo, fontsize=12, fontweight='bold', y=0.95)

    ax = plt.gca()
    ax.spines[:].set_color('dimgray')
    ax.tick_params(axis='both', colors='dimgray')
    ax.yaxis.label.set_color('dimgray')
    ax.xaxis.label.set_color('dimgray')

    plt.grid(True, linestyle='-', linewidth=0.4, color='lightgray')
    plt.xticks(np.arange(0, 24.5, 1))
    plt.xlabel("Hora del día")
    plt.ylabel("Velocidad (km/h)")

    # --- AJUSTAR ESPACIO VERTICAL AUTOMÁTICAMENTE ---

    # Valores mínimos y máximos reales del tramo
    vmin = min( grupo_tramo["p5"].min(), grupo_tramo["v85"].min() )
    vmax = max( grupo_tramo["p95"].max(), grupo_tramo["v85"].max() )
    
    # Margen del 10% (ajustable)
    padding = (vmax - vmin) * 0.2
    
    # Establecer límites ampliados
    plt.ylim(vmin - padding, vmax + padding)


    plt.legend(
        handles=legend_elements,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.25),
        ncol=len(periodos_ordenados),
        frameon=False,
        fontsize=9
    )

    plt.tight_layout()
    plt.show()

### Graficos diarios individuales

In [ ]:
#  GRÁFICOS INDIVIDUALES POR TRAMO Y PERIODO

print("\nGRÁFICOS INDIVIDUALES POR TRAMO Y PERIODO")

# Comprobar si hay info de velocidades límite disponible
usar_vlim_individual = 'usar_vlim' in globals() and usar_vlim and 'vlimites' in globals()

for tramo, grupo_tramo in estadisticas_horarias.groupby('Tramo'):
    print(f"\n=== Tramo: {tramo} ===\n")  # Agrupación visual por tramo

    for periodo, df in grupo_tramo.groupby('Periodo'):
        plt.figure(figsize=(10, 4))

        periodo_plano = periodo.replace('\n', ' - ')

        # ------------------------
        # Zona de percentiles 5%-95%
        # ------------------------
        plt.fill_between(
            df['HoraDecimal'],
            df['p5'],
            df['p95'],
            color='skyblue',
            alpha=0.35,
            label="Percentiles 5%-95%"
        )

        # ------------------------
        # Línea V85
        # ------------------------
        plt.plot(
            df['HoraDecimal'],
            df['v85'],
            color='blue',
            linewidth=2,
            label="V₈₅"
        )

        # ------------------------
        # Línea de velocidad límite (si existe)
        # ------------------------
        if usar_vlim_individual and tramo in vlimites:
            vlim = vlimites[tramo]

            # Línea roja horizontal
            plt.axhline(
                y=vlim,
                color="#CC0000",
                linewidth=1.0,
                linestyle='-'
            )

            # Etiqueta a la derecha, por debajo de la línea
            x_text = df['HoraDecimal'].max() - 0.1
            y_text = vlim - 1.2  # ajustar si quieres más/menos separación

            plt.text(
                x_text,
                y_text,
                f"V lim = {vlim:.1f} km/h",
                color="#CC0000",
                fontsize=7,
                ha="right",
                va="top"
            )

        # ------------------------
        # Títulos
        # ------------------------
        plt.suptitle(tramo, fontsize=12, fontweight='bold', color='black', y=0.95)
        plt.title(periodo_plano, fontsize=10, color='dimgray', y=1.02, x=0.47)

        # ------------------------
        # Estilos generales
        # ------------------------
        ax = plt.gca()
        ax.spines[:].set_color('dimgray')
        ax.tick_params(axis='both', colors='dimgray')
        ax.yaxis.label.set_color('dimgray')
        ax.xaxis.label.set_color('dimgray')

        # Cuadrícula
        plt.grid(True, linestyle='-', linewidth=0.4, color='lightgray')

        # Eje X
        plt.xticks(np.arange(0, 24.1, 1))
        plt.xlabel("Hora del día")
        plt.ylabel("Velocidad (km/h)")

       
        # Ajuste de rango vertical (padding)
        
        # Calcular mínimos y máximos reales del periodo
        vmin = min(df['p5'].min(), df['v85'].min())
        vmax = max(df['p95'].max(), df['v85'].max())

        # Tener en cuenta la vlim si existe
        if usar_vlim_individual and tramo in vlimites:
            vmin = min(vmin, vlimites[tramo])
            vmax = max(vmax, vlimites[tramo])

        padding = (vmax - vmin) * 0.20 if vmax > vmin else 5  # 20% de margen
        plt.ylim(vmin - padding, vmax + padding)

        
        # Leyenda
        
        plt.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.25),
            ncol=2,
            frameon=False,
            fontsize=9
        )

        plt.tight_layout()
        plt.show()